## Scalability Testing

This notebook is for testing purposes.

In [23]:
from simulator import Simulator
from simulator.config import DATA_PATH, Color
from simulator.helpers import clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.planner import AutoPlan, Plan
from simulator.visualizer import NoVisualizer, SimVehicle, Gazebo, GazMarker

clean()

## Simulation Positions

In [24]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241,alt=0,heading=0) 
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading) 

gcs_colors=[Color.RED,Color.ORANGE,Color.GREEN,Color.BLUE]
n_uavs_per_gcs = 1
side_len = 10
altitude = 5

base_homes = ENUPose.list([
    (i * 50, j * 3 * side_len, 0, 0)
    for i in range(len(gcs_colors))
    for j in range(n_uavs_per_gcs)
])

## Create Vehicles

In [25]:
sysids = range(1, len(base_homes)+1)
colors: list[Color]= []
for color in gcs_colors:
    colors.extend([color]*n_uavs_per_gcs)

vehs:list[SimVehicle] = []
for sysid, base_home, color in zip(sysids, base_homes, colors):
    auto_plan = AutoPlan.square_traj(
        side_len=side_len,
        alt=altitude,
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs_name=f'{color.name}_{color.emoji}',
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=Plan.create_square_path(side_len=side_len),
        model="iris",
    )
    vehs.append(veh)
    

## Visualizer

### Running Gazebo with No Visualization

In the original `uav-cyber-sim` source code, the `NoVisualization` visualizer used [ArduPilot's built in SITL simulator](ardupilot.org/dev/docs/using-sitl-for-ardupilot-testing.html#selecting-a-vehicle-frame-type) to run the simulations. However, unlike ArduPilot, PX4 does not have a built in simulator.

To enable the use of simulations with "no visualizations", `Gazebo` has been modified so it can run in headless mode. Thus, rather than [instantiating the `NoVisualizer` class](https://github.com/4belito/uav-cyber-sim/blob/main/2-simple_sim_iris.ipynb) as required the original `uav-cyber-sim` source code, you will instead pass `headless=True` when initializing `Gazebo` to run a simulation with no visualizations. This approach enables PX4 to utilize Gazebo's physics simulator when running simulations where it is preferred that no visualizations are shown (e.g. when running simulations with many drones).

In [26]:
gaz= Gazebo(gra_origin,world_path="simulator/visualizer/gazebo/worlds/runway.world",
            headless=False)
origin_gaz = GazMarker(name="origin",
                    group="origin",
                    pos=enu_origin.unpose(),
                    color=Color.WHITE)
gaz.markers.append(origin_gaz)

## Simulator

In [27]:
simulator = Simulator(
	visualizer=gaz,
	terminals=['gcs'],
	verbose=1,
)
for veh in vehs:
    simulator.add_vehicle(veh)

## Run

In [28]:
orac = simulator.launch()
orac.run()

/home/ubuntu/uav-cyber-sim/simulator/data


19:05:59 - Oracle ⚪ - INFO - 🖥️  Gazebo launched for realistic simulation and 3D visualization.
19:05:59 - Oracle ⚪ - INFO - 🚀 GCS RED_🟥 launched (PID 59391)
19:05:59 - Oracle ⚪ - INFO - 🚀 GCS ORANGE_🟧 launched (PID 59392)
19:05:59 - Oracle ⚪ - INFO - 🚀 GCS GREEN_🟩 launched (PID 59394)
# Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1000/at-spi/bus: No such file or directory
19:05:59 - Oracle ⚪ - INFO - 🚀 GCS BLUE_🟦 launched (PID 59395)
19:05:59 - Oracle ⚪ - INFO - 🏁 Starting Oracle with 4 vehicles and 4 GCSs
# Failed to load module "canberra-gtk-module"
# Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1000/at-spi/bus: No such file or directory
# Failed to load module "canberra-gtk-module"
# Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1000/at-spi/bus: No such file or directory
# Failed to load module "canberra-gtk-module"
# Failed to load module "canberra-gtk-module"
# Failed to load module "c

KeyboardInterrupt: 